In [1]:
import os 

In [2]:
os.chdir('../')

In [3]:
%pwd

'/Users/eshakunder/Desktop/TextSummarization/Text-Summarization-'

In [4]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelTrainerConfig:
    root_dir: Path
    data_path: Path
    model_ckpt : Path
    num_train_epochs : int
    warmup_steps : int
    per_device_train_batch_size : int
    per_device_eval_batch_size : int
    weight_decay : float
    logging_steps : int
    eval_strategy : str
    save_steps : float
    gradient_accumulation_steps : int
    fp16 : bool
    report_to : str

In [5]:
# change directory to your project folder
os.chdir("/Users/eshakunder/Desktop/TextSummarization/Text-Summarization-/src")

# confirm
print(os.getcwd())

/Users/eshakunder/Desktop/TextSummarization/Text-Summarization-/src


In [6]:
from textSummarizer.contants import *
from textSummarizer.utils.common import read_yaml, create_directories

class ConfigurationManager:
    def __init__(
            self,
            config_filepath = CONFIG_FILE_PATH,
            params_filepath = PARAMS_FILE_PATH):
            self.config = read_yaml(config_filepath)
            self.params = read_yaml(params_filepath)

            create_directories([self.config.artifacts_root])
    
    def get_model_trainer_config(self) -> ModelTrainerConfig:
        config = self.config.model_trainer
        params = self.params.TrainingArguments

        create_directories([config.root_dir])

        model_trainer = ModelTrainerConfig(
            root_dir = config.root_dir,
            data_path = config.data_path,
            model_ckpt = config.model_ckpt,
            num_train_epochs = params.num_train_epochs,
            warmup_steps = params.warmup_steps,
            per_device_train_batch_size = params.per_device_train_batch_size,
            per_device_eval_batch_size = params.per_device_eval_batch_size,
            weight_decay = params.weight_decay,
            logging_steps = params.logging_steps,
            eval_strategy = params.eval_strategy,
            save_steps = params.save_steps,
            gradient_accumulation_steps = params.gradient_accumulation_steps,
            fp16 = params.fp16,
            report_to = params.report_to
        )

        return model_trainer

CONFIG_FILE_PATH: config/config.yaml
PARAMS_FILE_PATH: params.yaml


In [7]:
import transformers
print(transformers.__version__)


4.56.1


In [8]:
from transformers import PreTrainedModel, AutoTokenizer, AutoModelForSeq2SeqLM

print("Transformers import works ✅")


: 

In [1]:
from transformers import TrainingArguments, Trainer
from transformers import pipeline , set_seed
from datasets import load_dataset , load_from_disk
from datasets import load_dataset
from transformers import AutoModelForSeq2SeqLM , AutoTokenizer
import torch 



ImportError: cannot import name 'PreTrainedModel' from 'transformers' (/opt/anaconda3/lib/python3.11/site-packages/transformers/__init__.py)

In [ ]:
class ModelTrainer:
    def __init__(self,config: ModelTrainerConfig):
        self.config = config

    def train(self):
        device = "cuda" if torch.cuda.is_available() else "cpu"
        tokenizer = AutoTokenizer.from_pretrained(self.config.model_ckpt)
        model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(self.config.model_ckpt).to(device)
        seq2seq_data_collator = DataCollatorForSeq2Seq(tokenizer, model=model_pegasus)

        #loading the dataset
        dataset_samsum_pt = load_from_disk(self.config.data_path)

        trainer_args = TrainingArguments(
            output_dir = self.config.root_dir,
            num_train_epochs = self.config.num_train_epochs,
            warmup_steps = self.config.warmup_steps,
            per_device_train_batch_size = self.config.per_device_train_batch_size,
            per_device_eval_batch_size = self.config.per_device_eval_batch_size,
            weight_decay = self.config.weight_decay,
            logging_steps = self.config.logging_steps,
            evaluation_strategy = self.config.eval_strategy,
            save_steps = int(self.config.save_steps),
            gradient_accumulation_steps = self.config.gradient_accumulation_steps,
            fp16 = self.config.fp16,
            report_to = self.config.report_to
        )
        trainer = Trainer(
          model = model_pegasus,
          args = trainer_args,
          tokenizer = tokenizer,
          data_collator = seq2seq_data_collator,
            train_dataset = dataset_samsum_pt["test"],
            eval_dataset = dataset_samsum_pt["validation"])
        
        trainer.train()

        model_pegasus.save_pretrained(os.path.join(self.config.root_dir,"pegasus-samsum-model"))

        tokenizer.save_pretrained(os.path.join(self.config.root_dir,"tokenizer"))

    

In [ ]:
try :
    config = ConfigurationManager()
    model_trainer_config = config.get_model_trainer_config()
    model_trainer_config = ModelTrainer(config = model_trainer_config)
    model_trainer_config.train()

except Exception as e:
    raise e